# Shelv: compare book vibe-search models

**Goal:** compare BM25 keyword search, multilingual E5-small, multilingual E5-base,
and BGE-M3 on the same cleaned catalog. We will measure operating performance and
prepare blind relevance judgments. **No model is declared best before evaluation.**

Your real request **“I need a book read peacefully”** is included verbatim.
The remaining 20 queries are draft examples, not collected user traffic.

You only need this notebook and **books_final.csv**. All experiment code and tests
are embedded below. Your CSV is uploaded to your Colab runtime when you choose it;
this notebook does not send book text to a hosted embedding API or mount Google Drive.

1. In Colab choose **Runtime → Change runtime type → GPU**.
2. Run steps 1–10 in order. The three model runs may take time on their first download.
3. Download the results ZIP before closing the runtime.
4. Review the development CSV, then continue with the optional scoring cells.

The earlier local input had 12,999 records, of which 7,626 had descriptions.
This notebook recalculates counts from your uploaded file rather than assuming them.
Colab GPU availability and runtime lifetime vary: [official FAQ](https://research.google.com/colaboratory/faq.html).


## 1. Install the model libraries

Use a fresh runtime. We pin the embedding library and Transformers versions, reuse
Colab's installed PyTorch, and save the complete installed package list. This
environment still needs to be validated by the actual runs; the notebook has not
been executed on a Colab GPU during its creation.


In [ ]:
#@title Install dependencies
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "sentence-transformers==5.1.2", "transformers==4.57.6",
    "huggingface-hub==0.36.2", "sentencepiece==0.2.1", "numpy>=1.26,<3"
], check=True)
print("Dependencies installed. If Colab requests a restart, restart and rerun from this cell.")


In [ ]:
#@title Check the GPU
import torch
ALLOW_CPU = False  # Set True only if you deliberately want a slower CPU experiment.
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, reconnect, and rerun.")


## 2. Set up the self-contained experiment

The next cell writes the benchmark and its tests into the temporary runtime.
The code is embedded, not fetched from an unversioned repository. Each experiment
records its input and code fingerprints. If you change queries/settings after a
run, choose a **new experiment name** instead of overwriting previous evidence.


In [ ]:
#@title Prepare experiment code (embedded for portability)
from pathlib import Path
import os
import json

EXPERIMENT_NAME = "comparison-colab-v1"
ROOT = Path("/content/shelv-vibe-search")
SCRIPTS = ROOT
PREPARED = ROOT / "artifacts/prepared"
EXPERIMENT = ROOT / "artifacts" / EXPERIMENT_NAME
for directory in (SCRIPTS, PREPARED, EXPERIMENT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(ROOT / "artifacts/model_cache")
os.environ["HF_HUB_DISABLE_XET"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
EMBEDDED_FILES = {'vibe_search.py': '"""Offline, evidence-preserving semantic retrieval experiment for Shelv."""\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nDEFAULT_INPUT = ROOT / \'data/books_final.csv\'\nDEFAULT_OUTPUT = ROOT / \'artifacts\'\nMODEL = \'intfloat/multilingual-e5-base\'\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef save_json(path, value):\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef document(row):\n    # No inferred mood labels, contaminated authors, or seller information.\n    parts = [f"Title: {row[\'title\'].strip()}"]\n    if row.get(\'subjects\', \'\').strip():\n        parts.append(f"Subjects: {row[\'subjects\'].strip()}")\n    parts.append(f"Description: {row[\'description\'].strip()}")\n    return \'\\n\'.join(parts)\n\n\ndef prepare(input_path, output):\n    csv.field_size_limit(10_000_000)\n    with input_path.open(encoding=\'utf-8-sig\', newline=\'\') as stream:\n        reader = csv.DictReader(stream)\n        required = {\'title\', \'description\', \'subjects\', \'language\', \'externalId\', \'sourceName\', \'sourceUrl\'}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(\'Missing required columns: \' + \', \'.join(sorted(required - set(reader.fieldnames or []))))\n        rows = list(reader)\n    documents, excluded = [], []\n    seen_ids = set()\n    for number, row in enumerate(rows, 1):\n        if None in row or any(v is None for v in row.values()):\n            raise ValueError(f\'Malformed CSV record {number}\')\n        # Include URL and record position because conflicting product IDs remain.\n        identity = json.dumps([row[\'sourceName\'], row[\'externalId\'], row[\'sourceUrl\'], number], ensure_ascii=False)\n        identity_value = row.get(\'record_id\', \'\').strip() or hashlib.sha256(identity.encode()).hexdigest()[:24]\n        if identity_value in seen_ids:\n            raise ValueError(f\'Duplicate record_id: {identity_value}\')\n        seen_ids.add(identity_value)\n        row = dict(row, record_id=identity_value)\n        if not row[\'title\'].strip() or not row[\'description\'].strip():\n            excluded.append(dict(row, exclusion_reason=\'missing_title_or_description\'))\n        else:\n            documents.append(dict(row, search_text=document(row)))\n    profile = {\'input\': str(input_path.resolve()), \'input_sha256\': digest(input_path),\n               \'input_rows\': len(rows), \'indexed_candidates\': len(documents),\n               \'excluded_rows\': len(excluded), \'sources\': {},\n               \'policy\': \'Nonblank title and description required; presence does not prove description quality.\',\n               \'status\': \'Data preparation only; retrieval relevance not evaluated.\'}\n    for source in sorted({r[\'sourceName\'] for r in rows}):\n        subset = [r for r in rows if r[\'sourceName\'] == source]\n        profile[\'sources\'][source] = {\n            \'rows\': len(subset),\n            \'missing_by_column\': {c: sum(not r[c].strip() for r in subset) for c in rows[0]},\n            \'languages\': dict(Counter(r[\'language\'] for r in subset)),\n            \'eligible\': sum(bool(r[\'title\'].strip() and r[\'description\'].strip()) for r in subset),\n        }\n    output.mkdir(parents=True, exist_ok=True)\n    save_json(output / \'documents.json\', documents)\n    save_json(output / \'excluded.json\', excluded)\n    save_json(output / \'profile.json\', profile)\n    return profile\n\n\ndef build(output, revision, batch_size):\n    if batch_size < 1:\n        raise ValueError(\'Batch size must be positive.\')\n    # Pin the remote model to an immutable commit, not a moving branch.\n    import re\n    if not re.fullmatch(r\'[0-9a-f]{40}\', revision):\n        raise ValueError(\'--revision must be the model repository full commit SHA.\')\n    import numpy as np\n    from sentence_transformers import SentenceTransformer\n    docs_path = output / \'documents.json\'\n    records = json.loads(docs_path.read_text(encoding=\'utf-8\'))\n    if not records:\n        raise ValueError(\'No eligible descriptions. Run prepare and inspect profile.json.\')\n    model = SentenceTransformer(MODEL, revision=revision)\n    texts = [\'passage: \' + r[\'search_text\'] for r in records]\n    lengths = [len(model.tokenizer(t, truncation=False, verbose=False)[\'input_ids\']) for t in texts]\n    started = time.perf_counter()\n    vectors = model.encode(texts, batch_size=batch_size, normalize_embeddings=True,\n                           convert_to_numpy=True, show_progress_bar=True).astype(\'float32\')\n    if vectors.ndim != 2 or vectors.shape[0] != len(records) or not np.isfinite(vectors).all():\n        raise ValueError(\'Invalid embedding matrix.\')\n    if not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4):\n        raise ValueError(\'Embeddings are not unit normalized.\')\n    np.save(output / \'embeddings.npy\', vectors)\n    save_json(output / \'truncation.json\', [\n        {\'record_id\': r[\'record_id\'], \'tokens\': n}\n        for r, n in zip(records, lengths) if n > model.max_seq_length\n    ])\n    manifest = {\'model\': MODEL, \'revision\': revision, \'max_tokens\': model.max_seq_length,\n                \'documents_sha256\': digest(docs_path), \'rows\': len(records),\n                \'embeddings_sha256\': digest(output / \'embeddings.npy\'),\n                \'dimensions\': vectors.shape[1], \'normalized\': True,\n                \'truncated_records\': sum(n > model.max_seq_length for n in lengths),\n                \'embedding_seconds\': time.perf_counter() - started,\n                \'python\': platform.python_version(),\n                \'packages\': {p: importlib.metadata.version(p) for p in (\'numpy\', \'sentence-transformers\', \'torch\', \'transformers\')},\n                \'status\': \'Candidate index; no relevance claim until human evaluation.\'}\n    save_json(output / \'manifest.json\', manifest)\n    return manifest\n\n\ndef rank(records, vectors, query_vector, k, language=None):\n    import numpy as np\n    if k < 1:\n        raise ValueError(\'k must be positive.\')\n    indices = [i for i, r in enumerate(records)\n               if not language or r[\'language\'].casefold() == language.casefold()]\n    if not indices:\n        return []\n    scores = vectors[indices] @ query_vector\n    ordered = np.argsort(-scores, kind=\'stable\')[:k]\n    return [dict(records[indices[i]], similarity=float(scores[i])) for i in ordered]\n\n\ndef search(output, query, k, language):\n    import numpy as np\n    if not query.strip():\n        raise ValueError(\'Query cannot be blank.\')\n    manifest = json.loads((output / \'manifest.json\').read_text())\n    if digest(output / \'documents.json\') != manifest[\'documents_sha256\']:\n        raise ValueError(\'Documents changed after indexing. Rebuild the index.\')\n    if digest(output / \'embeddings.npy\') != manifest[\'embeddings_sha256\']:\n        raise ValueError(\'Embedding file changed after indexing. Rebuild the index.\')\n    records = json.loads((output / \'documents.json\').read_text(encoding=\'utf-8\'))\n    vectors = np.load(output / \'embeddings.npy\', allow_pickle=False)\n    if vectors.shape != (len(records), manifest[\'dimensions\']) or not np.isfinite(vectors).all():\n        raise ValueError(\'Invalid index dimensions or values.\')\n    from sentence_transformers import SentenceTransformer\n    model = SentenceTransformer(manifest[\'model\'], revision=manifest[\'revision\'])\n    if len(model.tokenizer(\'query: \' + query)[\'input_ids\']) > manifest[\'max_tokens\']:\n        raise ValueError(\'Query exceeds model token limit; shorten it.\')\n    started = time.perf_counter()\n    query_vector = model.encode([\'query: \' + query.strip()], normalize_embeddings=True,\n                                convert_to_numpy=True)[0]\n    hits = rank(records, vectors, query_vector, k, language)\n    return {\'query\': query, \'language_filter\': language,\n            \'search_seconds_excluding_model_load\': time.perf_counter() - started,\n            \'notice\': \'Nearest matches, not verified relevance. Scores are not probabilities.\',\n            \'results\': [{key: r[key] for key in (\'record_id\', \'title\', \'description\',\n                        \'language\', \'sourceName\', \'sourceUrl\', \'similarity\')} for r in hits]}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    for command in (\'prepare\', \'build\', \'search\'):\n        subparser = sub.add_parser(command)\n        subparser.add_argument(\'--output\', type=Path, default=DEFAULT_OUTPUT)\n        if command == \'prepare\':\n            subparser.add_argument(\'--input\', type=Path, default=DEFAULT_INPUT)\n        elif command == \'build\':\n            subparser.add_argument(\'--revision\', required=True)\n            subparser.add_argument(\'--batch-size\', type=int, default=16)\n        else:\n            subparser.add_argument(\'query\')\n            subparser.add_argument(\'--language\')\n            subparser.add_argument(\'--k\', type=int, default=5)\n    args = parser.parse_args()\n    try:\n        if args.command == \'prepare\':\n            result = prepare(args.input, args.output)\n        elif args.command == \'build\':\n            result = build(args.output, args.revision, args.batch_size)\n        else:\n            result = search(args.output, args.query, args.k, args.language)\n        print(json.dumps(result, ensure_ascii=False, indent=2))\n    except (ValueError, FileNotFoundError, ImportError) as error:\n        parser.exit(1, f\'{error}\\n\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'benchmark.py': '"""Reproducible retrieval comparison; no relevance labels are invented."""\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport resource\nimport statistics\nimport sys\nimport time\nimport unicodedata\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom vibe_search import DEFAULT_OUTPUT, digest, save_json\n\nHERE = Path(__file__).resolve().parent\nCANDIDATES = {\n    \'e5-small\': {\'model\': \'intfloat/multilingual-e5-small\', \'query_prefix\': \'query: \', \'passage_prefix\': \'passage: \'},\n    \'e5-base\': {\'model\': \'intfloat/multilingual-e5-base\', \'query_prefix\': \'query: \', \'passage_prefix\': \'passage: \'},\n    \'bge-m3\': {\'model\': \'BAAI/bge-m3\', \'query_prefix\': \'\', \'passage_prefix\': \'\'},\n}\n\n\ndef tokens(text):\n    # Preserve Tamil combining marks. No English-only token regex or stemming.\n    text = unicodedata.normalize(\'NFC\', text).casefold()\n    return \'\'.join(c if unicodedata.category(c)[0] in \'LMN\' else \' \' for c in text).split()\n\n\nclass BM25:\n    def __init__(self, texts, k1=1.5, b=0.75):\n        self.k1, self.b = k1, b\n        counts = [Counter(tokens(t)) for t in texts]\n        self.lengths = np.array([sum(c.values()) for c in counts], dtype=float)\n        self.average = float(self.lengths.mean()) if len(counts) else 0\n        self.postings = defaultdict(list)\n        for i, count in enumerate(counts):\n            for term, frequency in count.items():\n                self.postings[term].append((i, frequency))\n\n    def score(self, query):\n        result = np.zeros(len(self.lengths))\n        if not self.average:\n            return result\n        for term in set(tokens(query)):\n            postings = self.postings.get(term, [])\n            if not postings:\n                continue\n            indices, frequencies = np.array(postings).T\n            idf = math.log(1 + (len(result) - len(postings) + 0.5) / (len(postings) + 0.5))\n            denominator = frequencies + self.k1 * (1 - self.b + self.b * self.lengths[indices] / self.average)\n            result[indices] += idf * frequencies * (self.k1 + 1) / denominator\n        return result\n\n\ndef select(records, scores, query, k, positive_only=False):\n    eligible = np.array([i for i, row in enumerate(records)\n                         if not query[\'language_filter\'] or row[\'language\'].casefold() == query[\'language_filter\'].casefold()], dtype=int)\n    if positive_only:\n        eligible = eligible[scores[eligible] > 0]\n    order = np.argsort(-scores[eligible], kind=\'stable\')[:k]\n    return [{\'record_id\': records[i][\'record_id\'], \'score\': float(scores[i]), \'rank\': rank}\n            for rank, i in enumerate(eligible[order], 1)]\n\n\ndef experiment(output, documents, queries_path, max_tokens):\n    output.mkdir(parents=True, exist_ok=True)\n    records = json.loads(documents.read_text(encoding=\'utf-8\'))\n    queries = json.loads(queries_path.read_text(encoding=\'utf-8\'))\n    if not records or len({r[\'record_id\'] for r in records}) != len(records):\n        raise ValueError(\'Corpus must contain unique record IDs and at least one record.\')\n    if not queries or len({q[\'id\'] for q in queries}) != len(queries):\n        raise ValueError(\'Queries must have unique IDs.\')\n    for q in queries:\n        if q[\'split\'] not in (\'dev\', \'test\') or not q[\'query\'].strip():\n            raise ValueError(\'Invalid query or split.\')\n    contract = {\'documents_sha256\': digest(documents), \'queries_sha256\': digest(queries_path),\n                \'code_sha256\': digest(Path(__file__)), \'max_tokens\': max_tokens,\n                \'rows\': len(records), \'query_count\': len(queries),\n                \'query_origins\': dict(Counter(q.get(\'origin\', \'draft_synthetic\') for q in queries)),\n                \'content_policy\': \'same title/subjects/description per record; model tokenizers may truncate differently\',\n                \'retrieval_depth\': 10, \'evaluation_depth\': 5,\n                \'status\': \'Unjudged experiment; no model winner established.\'}\n    path = output / \'experiment.json\'\n    if path.exists() and json.loads(path.read_text()) != contract:\n        raise ValueError(\'Experiment inputs/code/settings changed. Choose a new --output directory.\')\n    save_json(path, contract)\n    save_json(output / \'corpus.json\', records)\n    save_json(output / \'queries.json\', queries)\n    return records, queries\n\n\ndef run(args):\n    records, queries = experiment(args.output, args.documents, args.queries, args.max_tokens)\n    run_path = args.output / (args.candidate + \'.json\')\n    if run_path.exists():\n        print(f\'{args.candidate}: existing completed run preserved. Use a new experiment to rerun.\')\n        return\n    config = CANDIDATES.get(args.candidate)\n    run_info = {\'candidate\': args.candidate, \'python\': platform.python_version(),\n                \'platform\': platform.platform(), \'rows\': len(records), \'queries\': [],\n                \'packages\': {}, \'max_tokens\': args.max_tokens, \'relevance_status\': \'unjudged\'}\n    for package in (\'numpy\', \'sentence-transformers\', \'torch\', \'transformers\', \'huggingface-hub\'):\n        try:\n            run_info[\'packages\'][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    started = time.perf_counter()\n    if config is None:\n        scorer = BM25([r[\'search_text\'] for r in records])\n        run_info.update(index_seconds=time.perf_counter() - started, device=\'cpu\', k1=1.5, b=0.75,\n                        passage_tokens=\'full text; Unicode words, no stemming\', truncated_records=0)\n        score_query = lambda q: scorer.score(q)\n        synchronize = lambda: None\n    else:\n        # Cache locally in the ignored experiment data tree, not a global directory.\n        os.environ.setdefault(\'HF_HOME\', str(DEFAULT_OUTPUT / \'model_cache\'))\n        os.environ.setdefault(\'HF_HUB_DISABLE_XET\', \'1\')\n        os.environ.setdefault(\'TOKENIZERS_PARALLELISM\', \'false\')\n        import torch\n        from huggingface_hub import HfApi\n        from sentence_transformers import SentenceTransformer\n        torch.set_num_threads(4)\n        device = \'mps\' if torch.backends.mps.is_available() else \'cuda\' if torch.cuda.is_available() else \'cpu\'\n        if device == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n            run_info[\'gpu_name\'] = torch.cuda.get_device_name()\n            run_info[\'cuda_version\'] = torch.version.cuda\n        revision_path = args.output / (args.candidate + \'-revision.json\')\n        if revision_path.exists():\n            revision = json.loads(revision_path.read_text())[\'revision\']\n        else:\n            revision = HfApi().model_info(config[\'model\']).sha\n            save_json(revision_path, dict(config, revision=revision))\n        print(f"Loading {config[\'model\']} at {revision} on {device}", flush=True)\n        model = SentenceTransformer(config[\'model\'], revision=revision, device=device, trust_remote_code=False)\n        model.max_seq_length = args.max_tokens\n        run_info.update(config, revision=revision, device=device, load_seconds=time.perf_counter() - started,\n                        batch_size=args.batch_size, precision=\'float32\')\n        def synchronize():\n            if device == \'mps\':\n                torch.mps.synchronize()\n            elif device == \'cuda\':\n                torch.cuda.synchronize()\n        texts = [config[\'passage_prefix\'] + r[\'search_text\'] for r in records]\n        lengths = [len(model.tokenizer(t, truncation=False, verbose=False)[\'input_ids\']) for t in texts]\n        run_info[\'truncated_records\'] = sum(n > args.max_tokens for n in lengths)\n        save_json(args.output / (args.candidate + \'-truncation.json\'),\n                  [{\'record_id\': r[\'record_id\'], \'tokens\': n} for r, n in zip(records, lengths) if n > args.max_tokens])\n        vector_path = args.output / (args.candidate + \'.npy\')\n        progress_path = args.output / (args.candidate + \'-progress.json\')\n        progress = json.loads(progress_path.read_text()) if progress_path.exists() else {\'completed\': 0, \'encoding_seconds\': 0}\n        dimension = model.get_sentence_embedding_dimension()\n        if progress[\'completed\']:\n            vectors = np.lib.format.open_memmap(vector_path, mode=\'r+\')\n            if vectors.shape != (len(records), dimension):\n                raise ValueError(\'Partial index shape does not match experiment.\')\n        else:\n            vectors = np.lib.format.open_memmap(vector_path, mode=\'w+\', dtype=\'float32\', shape=(len(records), dimension))\n        for start in range(progress[\'completed\'], len(records), 128):\n            stop = min(start + 128, len(records))\n            synchronize()\n            before = time.perf_counter()\n            batch = model.encode(texts[start:stop], batch_size=args.batch_size, normalize_embeddings=True,\n                                 convert_to_numpy=True, show_progress_bar=False)\n            synchronize()\n            if not np.isfinite(batch).all() or not np.allclose(np.linalg.norm(batch, axis=1), 1, atol=1e-4):\n                raise ValueError(\'Invalid embeddings.\')\n            vectors[start:stop] = batch\n            vectors.flush()\n            progress[\'encoding_seconds\'] += time.perf_counter() - before\n            progress[\'completed\'] = stop\n            save_json(progress_path, progress)\n            print(f\'{args.candidate}: {stop}/{len(records)} books encoded\', flush=True)\n        run_info.update(index_seconds=progress[\'encoding_seconds\'], dimensions=dimension,\n                        embedding_file_bytes=vector_path.stat().st_size, vectors_sha256=digest(vector_path))\n        def score_query(q):\n            text = config[\'query_prefix\'] + q\n            if len(model.tokenizer(text)[\'input_ids\']) > args.max_tokens:\n                raise ValueError(\'Query exceeds configured token budget.\')\n            vector = model.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0]\n            return vectors @ vector\n    # Untimed warmup, then repeated single-query measurements excluding model load.\n    score_query(queries[0][\'query\'])\n    synchronize()\n    for query in queries:\n        timings = []\n        for _ in range(3):\n            synchronize()\n            before = time.perf_counter()\n            scores = score_query(query[\'query\'])\n            hits = select(records, scores, query, 10, positive_only=config is None)\n            synchronize()\n            timings.append(time.perf_counter() - before)\n        run_info[\'queries\'].append({\'id\': query[\'id\'], \'latency_median_ms\': statistics.median(timings) * 1000,\n                                    \'hits\': hits})\n    run_info[\'query_p50_ms\'] = float(np.median([q[\'latency_median_ms\'] for q in run_info[\'queries\']]))\n    run_info[\'query_p95_ms\'] = float(np.percentile([q[\'latency_median_ms\'] for q in run_info[\'queries\']], 95))\n    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    run_info[\'peak_process_rss_bytes\'] = rss if sys.platform == \'darwin\' else rss * 1024\n    run_info[\'memory_note\'] = \'Process peak RSS; GPU allocations not fully represented; includes loading/tokenization.\'\n    if config and device == \'mps\':\n        run_info[\'mps_current_allocated_bytes\'] = torch.mps.current_allocated_memory()\n        run_info[\'mps_current_driver_bytes\'] = torch.mps.driver_allocated_memory()\n    elif config and device == \'cuda\':\n        run_info[\'cuda_peak_allocated_bytes\'] = torch.cuda.max_memory_allocated()\n        run_info[\'cuda_peak_reserved_bytes\'] = torch.cuda.max_memory_reserved()\n    save_json(run_path, run_info)\n    print(f\'Saved completed run: {run_path}\', flush=True)\n\n\ndef completed_runs(output):\n    return [json.loads((output / (c + \'.json\')).read_text())\n            for c in [\'bm25\', *CANDIDATES] if (output / (c + \'.json\')).exists()]\n\n\ndef pool(args):\n    queries = json.loads((args.output / \'queries.json\').read_text(encoding=\'utf-8\'))\n    records = {r[\'record_id\']: r for r in json.loads((args.output / \'corpus.json\').read_text(encoding=\'utf-8\'))}\n    runs = completed_runs(args.output)\n    if {r[\'candidate\'] for r in runs} != {\'bm25\', *CANDIDATES}:\n        raise ValueError(\'All four candidates must finish before pooling to avoid a biased review pool.\')\n    pooled = defaultdict(set)\n    for run in runs:\n        for q in run[\'queries\']:\n            pooled[q[\'id\']].update(h[\'record_id\'] for h in q[\'hits\'])\n    for split in (\'dev\', \'test\'):\n        path = args.output / (\'review_\' + split + \'.csv\')\n        if path.exists():\n            raise ValueError(f\'{path} exists; preserving reviewer work. Use a new experiment.\')\n    for split in (\'dev\', \'test\'):\n        rows = []\n        for q in queries:\n            if q[\'split\'] != split:\n                continue\n            ids = sorted(pooled[q[\'id\']])\n            random.Random(q[\'id\']).shuffle(ids)\n            for identity in ids:\n                r = records[identity]\n                rows.append({\'query_id\': q[\'id\'], \'query\': q[\'query\'], \'language_filter\': q[\'language_filter\'] or \'\',\n                             \'intent\': q[\'intent\'], \'record_id\': identity, \'title\': r[\'title\'],\n                             \'description\': r[\'description\'], \'book_language\': r[\'language\'], \'source_url\': r[\'sourceUrl\'],\n                             \'grade\': \'\', \'evidence\': \'\', \'reviewer\': \'\'})\n        fields = [\'query_id\', \'query\', \'language_filter\', \'intent\', \'record_id\', \'title\', \'description\', \'book_language\', \'source_url\', \'grade\', \'evidence\', \'reviewer\']\n        with (args.output / (\'review_\' + split + \'.csv\')).open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n        print(f\'{split}: {len(rows)} unique query/book pairs for blind review\')\n\n\ndef dcg(grades):\n    return sum((2 ** grade - 1) / math.log2(rank + 2) for rank, grade in enumerate(grades))\n\n\ndef evaluate(args):\n    queries = {q[\'id\']: q for q in json.loads((args.output / \'queries.json\').read_text()) if q[\'split\'] == args.split}\n    runs = completed_runs(args.output)\n    pool_ids = defaultdict(set)\n    for run in runs:\n        for q in run[\'queries\']:\n            if q[\'id\'] in queries:\n                pool_ids[q[\'id\']].update(h[\'record_id\'] for h in q[\'hits\'])\n    judgments, seen = {}, set()\n    with args.judgments.open(encoding=\'utf-8-sig\', newline=\'\') as stream:\n        for row in csv.DictReader(stream):\n            key = (row[\'query_id\'], row[\'record_id\'])\n            if key in seen:\n                raise ValueError(f\'Duplicate judgment: {key}\')\n            seen.add(key)\n            if key[0] not in queries or key[1] not in pool_ids[key[0]]:\n                raise ValueError(f\'Judgment is not in this experiment/split: {key}\')\n            if row[\'grade\'].strip() in (\'\', \'U\'):\n                continue\n            if row[\'grade\'] not in (\'0\', \'1\', \'2\') or not row[\'evidence\'].strip() or not row[\'reviewer\'].strip():\n                raise ValueError(\'Each grade must be 0/1/2 with evidence and reviewer; U means insufficient evidence.\')\n            judgments[key] = int(row[\'grade\'])\n    complete = {q for q, ids in pool_ids.items() if ids and all((q, i) in judgments for i in ids)}\n    results = []\n    for run in runs:\n        per_query = []\n        for result in run[\'queries\']:\n            qid = result[\'id\']\n            if qid not in complete:\n                continue\n            grades = [judgments[qid, h[\'record_id\']] for h in result[\'hits\'][:5]]\n            ideal = sorted([judgments[qid, i] for i in pool_ids[qid]], reverse=True)[:5]\n            denominator = dcg(ideal)\n            per_query.append({\'query_id\': qid, \'query_language\': queries[qid][\'query_language\'],\n                              \'intent\': queries[qid][\'intent\'], \'precision_at_5\': sum(g == 2 for g in grades) / 5,\n                              \'pooled_ndcg_at_5\': dcg(grades) / denominator if denominator else None})\n        ordinary = [q for q in per_query if q[\'intent\'] != \'out_of_domain\']\n        ndcgs = [q[\'pooled_ndcg_at_5\'] for q in ordinary if q[\'pooled_ndcg_at_5\'] is not None]\n        results.append({\'candidate\': run[\'candidate\'], \'fully_judged_queries\': len(per_query),\n                        \'precision_at_5\': statistics.mean(q[\'precision_at_5\'] for q in ordinary) if ordinary else None,\n                        \'pooled_ndcg_at_5\': statistics.mean(ndcgs) if ndcgs else None,\n                        \'query_p50_ms\': run[\'query_p50_ms\'], \'query_p95_ms\': run[\'query_p95_ms\'],\n                        \'index_seconds\': run[\'index_seconds\'], \'peak_process_rss_bytes\': run[\'peak_process_rss_bytes\'],\n                        \'per_query\': per_query})\n    report = {\'split\': args.split, \'judged_pairs\': len(judgments),\n              \'pooled_pairs\': sum(map(len, pool_ids.values())), \'fully_judged_queries\': len(complete),\n              \'total_queries\': len(queries), \'results\': results,\n              \'decision\': \'No automatic winner: assess relevance, language slices, uncertainty and operating constraints.\',\n              \'limitations\': \'Pilot with one user-provided query and proposed synthetic queries; not representative of real query traffic. Unjudged/U pairs never treated as irrelevant. nDCG is limited to the pooled candidates, not exhaustive catalog relevance. OOD queries excluded from aggregate relevance; inspect separately.\'}\n    save_json(args.output / (\'evaluation_\' + args.split + \'.json\'), report)\n    lines = [\'# Model comparison\', \'\', f"Split: {args.split}. Fully judged queries: {len(complete)}/{len(queries)}.", \'\',\n             \'| Candidate | Precision@5 | Pooled nDCG@5 | Median query ms | p95 query ms | Index seconds | Peak process MiB |\',\n             \'| --- | ---: | ---: | ---: | ---: | ---: | ---: |\']\n    for r in results:\n        precision = \'Not judged\' if r[\'precision_at_5\'] is None else f"{r[\'precision_at_5\']:.3f}"\n        ndcg = \'Not judged\' if r[\'pooled_ndcg_at_5\'] is None else f"{r[\'pooled_ndcg_at_5\']:.3f}"\n        lines.append(f"| {r[\'candidate\']} | {precision} | {ndcg} | {r[\'query_p50_ms\']:.1f} | {r[\'query_p95_ms\']:.1f} | {r[\'index_seconds\']:.1f} | {r[\'peak_process_rss_bytes\']/2**20:.1f} |")\n    lines += [\'\', report[\'decision\'], \'\', report[\'limitations\'], \'\',\n              \'Latency excludes model loading and is based on three warm measurements per query. Hardware timings are pilot observations, not production SLA measurements. Peak process RSS is not total GPU memory.\']\n    (args.output / (\'comparison_\' + args.split + \'.md\')).write_text(\'\\n\'.join(lines) + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    for command in (\'run\', \'pool\', \'evaluate\'):\n        child = sub.add_parser(command)\n        child.add_argument(\'--output\', type=Path, default=DEFAULT_OUTPUT / \'benchmark-v1\')\n        if command == \'run\':\n            child.add_argument(\'--candidate\', choices=[\'bm25\', *CANDIDATES], required=True)\n            child.add_argument(\'--documents\', type=Path, default=DEFAULT_OUTPUT / \'documents.json\')\n            child.add_argument(\'--queries\', type=Path, default=HERE / \'benchmark_queries.json\')\n            child.add_argument(\'--max-tokens\', type=int, default=512, choices=range(32, 513), metavar=\'32..512\')\n            child.add_argument(\'--batch-size\', type=int, default=8)\n        elif command == \'evaluate\':\n            child.add_argument(\'--judgments\', type=Path, required=True)\n            child.add_argument(\'--split\', choices=[\'dev\', \'test\'], default=\'dev\')\n    args = parser.parse_args()\n    if getattr(args, \'batch_size\', 1) < 1:\n        parser.error(\'Batch size must be positive.\')\n    {\'run\': run, \'pool\': pool, \'evaluate\': evaluate}[args.command](args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'benchmark_queries.json': '[\n  {"id":"user-en-01","split":"dev","query":"I need a book read peacefully","language_filter":null,"query_language":"English","intent":"vibe","origin":"user_provided"},\n  {"id":"dev-en-01","split":"dev","query":"A comforting story about friendship and starting over","language_filter":null,"query_language":"English","intent":"vibe"},\n  {"id":"dev-ta-01","split":"dev","query":"நட்பையும் நம்பிக்கையையும் பற்றிய மனதுக்கு ஆறுதல் தரும் கதை","language_filter":null,"query_language":"Tamil","intent":"vibe"},\n  {"id":"dev-en-02","split":"dev","query":"A slow reflective story about loneliness and the meaning of life","language_filter":null,"query_language":"English","intent":"vibe"},\n  {"id":"dev-ta-02","split":"dev","query":"தனிமையையும் வாழ்க்கையின் அர்த்தத்தையும் பற்றிய அமைதியான கதை","language_filter":null,"query_language":"Tamil","intent":"vibe"},\n  {"id":"dev-en-03","split":"dev","query":"A tense story about betrayal and revenge","language_filter":null,"query_language":"English","intent":"vibe"},\n  {"id":"dev-ta-03","split":"dev","query":"துரோகத்தையும் பழிவாங்குவதையும் பற்றிய விறுவிறுப்பான கதை","language_filter":null,"query_language":"Tamil","intent":"vibe"},\n  {"id":"dev-en-04","split":"dev","query":"An inspiring account of a woman overcoming social barriers","language_filter":"Tamil","query_language":"English","intent":"vibe"},\n  {"id":"dev-ta-04","split":"dev","query":"சமூகத் தடைகளைத் தாண்டி முன்னேறும் பெண்ணின் ஊக்கமளிக்கும் வாழ்க்கைக் கதை","language_filter":"Tamil","query_language":"Tamil","intent":"vibe"},\n  {"id":"dev-en-05","split":"dev","query":"A comforting story without violence","language_filter":null,"query_language":"English","intent":"constraint"},\n  {"id":"dev-ta-05","split":"dev","query":"வன்முறை இல்லாத மனதுக்கு ஆறுதல் தரும் கதை","language_filter":null,"query_language":"Tamil","intent":"constraint"},\n  {"id":"dev-en-06","split":"dev","query":"How to repair a broken dishwasher pump","language_filter":null,"query_language":"English","intent":"out_of_domain"},\n  {"id":"dev-ta-06","split":"dev","query":"பழுதான பாத்திரம் கழுவும் இயந்திரத்தின் பம்பை சரிசெய்வது எப்படி","language_filter":null,"query_language":"Tamil","intent":"out_of_domain"},\n  {"id":"test-en-01","split":"test","query":"A bittersweet story about returning home and childhood memories","language_filter":null,"query_language":"English","intent":"vibe"},\n  {"id":"test-ta-01","split":"test","query":"சொந்த ஊருக்குத் திரும்புவதையும் குழந்தைப் பருவ நினைவுகளையும் பற்றிய இனிமையும் சோகமும் கலந்த கதை","language_filter":null,"query_language":"Tamil","intent":"vibe"},\n  {"id":"test-en-02","split":"test","query":"A funny lighthearted book that helps me unwind","language_filter":null,"query_language":"English","intent":"vibe"},\n  {"id":"test-ta-02","split":"test","query":"மன அழுத்தத்தைக் குறைக்கும் நகைச்சுவையான இலகுவான புத்தகம்","language_filter":null,"query_language":"Tamil","intent":"vibe"},\n  {"id":"test-en-03","split":"test","query":"A moving story about displacement and belonging","language_filter":"Tamil","query_language":"English","intent":"vibe"},\n  {"id":"test-ta-03","split":"test","query":"இடம்பெயர்வையும் சொந்த மண்ணின் ஏக்கத்தையும் பற்றிய மனதைத் தொடும் கதை","language_filter":"Tamil","query_language":"Tamil","intent":"vibe"},\n  {"id":"test-en-04","split":"test","query":"A gentle love story with a happy ending","language_filter":null,"query_language":"English","intent":"constraint"},\n  {"id":"test-ta-04","split":"test","query":"மகிழ்ச்சியான முடிவைக் கொண்ட மென்மையான காதல் கதை","language_filter":null,"query_language":"Tamil","intent":"constraint"}\n]\n', 'test_vibe_search.py': "import csv\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nimport numpy as np\nfrom vibe_search import document, prepare, rank\n\n\nclass VibeSearchTests(unittest.TestCase):\n    def test_document_preserves_tamil_and_excludes_author_contamination(self):\n        text = document({'title': 'நட்பு', 'subjects': 'Novel',\n                         'description': 'A story of friendship.', 'authors': 'unreliable tags'})\n        self.assertIn('நட்பு', text)\n        self.assertNotIn('unreliable', text)\n\n    def test_missing_descriptions_are_accounted_for_and_input_unchanged(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            path = root / 'books.csv'\n            fields = ['title', 'description', 'subjects', 'language', 'externalId', 'sourceName', 'sourceUrl']\n            with path.open('w', newline='') as stream:\n                writer = csv.DictWriter(stream, fieldnames=fields)\n                writer.writeheader()\n                for description in (' friendship ', '  '):\n                    writer.writerow(dict.fromkeys(fields, '') | {'title': 'Book', 'description': description, 'sourceName': 'Shop'})\n            original = path.read_bytes()\n            profile = prepare(path, root / 'out')\n            self.assertEqual((profile['input_rows'], profile['indexed_candidates'], profile['excluded_rows']), (2, 1, 1))\n            self.assertEqual(path.read_bytes(), original)\n            docs = (root / 'out/documents.json').read_bytes()\n            prepare(path, root / 'out')\n            self.assertEqual(docs, (root / 'out/documents.json').read_bytes())\n\n    def test_imported_ids_survive_reordering_and_duplicates_fail(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            path = root / 'books.csv'\n            fields = ['record_id', 'title', 'description', 'subjects', 'language', 'externalId', 'sourceName', 'sourceUrl']\n            def write(ids):\n                with path.open('w', newline='') as stream:\n                    writer = csv.DictWriter(stream, fieldnames=fields)\n                    writer.writeheader()\n                    for identity in ids:\n                        writer.writerow(dict.fromkeys(fields, '') | {'record_id': identity, 'title': 'Book', 'description': 'Story'})\n            write(['book-b', 'book-a'])\n            prepare(path, root / 'out')\n            rows = json.loads((root / 'out/documents.json').read_text())\n            self.assertEqual([r['record_id'] for r in rows], ['book-b', 'book-a'])\n            write(['book-a', 'book-a'])\n            with self.assertRaisesRegex(ValueError, 'Duplicate record_id'):\n                prepare(path, root / 'out')\n\n    def test_filter_applied_before_top_k(self):\n        rows = [{'language': 'English', 'id': 'a'}, {'language': 'Tamil', 'id': 'b'}]\n        vectors = np.array([[1., 0.], [0.6, 0.8]])\n        result = rank(rows, vectors, np.array([1., 0.]), 1, 'tamil')\n        self.assertEqual(result[0]['id'], 'b')\n        self.assertAlmostEqual(result[0]['similarity'], 0.6)\n        self.assertEqual(rank(rows, vectors, np.array([1., 0.]), 5, 'French'), [])\n\n    def test_ranking_and_stable_ties(self):\n        rows = [{'language': 'Tamil', 'id': i} for i in range(3)]\n        vectors = np.array([[0., 1.], [1., 0.], [1., 0.]])\n        hits = rank(rows, vectors, np.array([1., 0.]), 5)\n        self.assertEqual([r['id'] for r in hits], [1, 2, 0])\n        with self.assertRaises(ValueError):\n            rank(rows, vectors, np.array([1., 0.]), 0)\n\n\nif __name__ == '__main__':\n    unittest.main()\n", 'test_benchmark.py': 'import argparse\nimport contextlib\nimport io\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom benchmark import BM25, dcg, evaluate, experiment, select, tokens\n\n\nclass BenchmarkTests(unittest.TestCase):\n    def test_tamil_words_keep_combining_marks(self):\n        self.assertEqual(tokens(\'நட்பு, நம்பிக்கை!\'), [\'நட்பு\', \'நம்பிக்கை\'])\n\n    def test_lexical_baseline_does_not_return_zero_score_matches(self):\n        scorer = BM25([\'friendship and hope\', \'crime and revenge\'])\n        scores = scorer.score(\'friendship\')\n        self.assertGreater(scores[0], scores[1])\n        records = [{\'record_id\': \'a\', \'language\': \'Tamil\'}, {\'record_id\': \'b\', \'language\': \'English\'}]\n        self.assertEqual(select(records, scorer.score(\'dishwasher\'), {\'language_filter\': None}, 5, True), [])\n\n    def test_language_filter_before_selection(self):\n        records = [{\'record_id\': \'a\', \'language\': \'Tamil\'}, {\'record_id\': \'b\', \'language\': \'English\'}]\n        hits = select(records, np.array([.5, .9]), {\'language_filter\': \'Tamil\'}, 1)\n        self.assertEqual(hits[0][\'record_id\'], \'a\')\n\n    def test_changed_corpus_rejected(self):\n        with tempfile.TemporaryDirectory() as d:\n            root = Path(d)\n            docs, queries = root / \'docs.json\', root / \'queries.json\'\n            docs.write_text(\'[{"record_id":"a"}]\')\n            queries.write_text(\'[{"id":"q","split":"dev","query":"hope"}]\')\n            experiment(root / \'run\', docs, queries, 512)\n            docs.write_text(\'[{"record_id":"b"}]\')\n            with self.assertRaises(ValueError):\n                experiment(root / \'run\', docs, queries, 512)\n\n    def test_dcg_rewards_relevant_book_first(self):\n        self.assertGreater(dcg([2, 0]), dcg([0, 2]))\n\n    def test_unjudged_is_not_irrelevant_and_full_pool_required(self):\n        with tempfile.TemporaryDirectory() as d:\n            root = Path(d)\n            (root / \'queries.json\').write_text(json.dumps([\n                {\'id\': \'q\', \'split\': \'dev\', \'query_language\': \'English\', \'intent\': \'vibe\'}]))\n            (root / \'bm25.json\').write_text(json.dumps({\n                \'candidate\': \'bm25\', \'query_p50_ms\': 1, \'query_p95_ms\': 2,\n                \'index_seconds\': 1, \'peak_process_rss_bytes\': 1024,\n                \'queries\': [{\'id\': \'q\', \'hits\': [{\'record_id\': \'a\'}, {\'record_id\': \'b\'}]}]}))\n            judgments = root / \'labels.csv\'\n            header = \'query_id,record_id,grade,evidence,reviewer\\n\'\n            judgments.write_text(header + \'q,a,2,Friendship,Reviewer\\nq,b,,,\\n\')\n            args = argparse.Namespace(output=root, judgments=judgments, split=\'dev\')\n            with contextlib.redirect_stdout(io.StringIO()):\n                evaluate(args)\n            report = json.loads((root / \'evaluation_dev.json\').read_text())\n            self.assertEqual(report[\'fully_judged_queries\'], 0)\n            self.assertIsNone(report[\'results\'][0][\'precision_at_5\'])\n            judgments.write_text(header + \'q,a,2,Friendship,Reviewer\\nq,b,0,Unrelated,Reviewer\\n\')\n            with contextlib.redirect_stdout(io.StringIO()):\n                evaluate(args)\n            report = json.loads((root / \'evaluation_dev.json\').read_text())\n            self.assertAlmostEqual(report[\'results\'][0][\'precision_at_5\'], .2)\n            self.assertAlmostEqual(report[\'results\'][0][\'pooled_ndcg_at_5\'], 1)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'README.md': '# Shelv Vibe Search\n\nStandalone Python retrieval experiment for Shelv: find books using natural-language mood and theme requests. Shelv is the parent product; this project owns offline retrieval and evaluation. It does not yet provide an API or connect to live seller inventory. No model has been selected as the winner.\n\n## Essential files\n\n- `vibe_search.py`: prepare catalog text, build an E5-base prototype index, search it.\n- `benchmark.py`: compare BM25, multilingual-E5-small, multilingual-E5-base, and BGE-M3; generate review sheets and score judgments.\n- `benchmark_queries.json`: 21 queries, 13 development / 8 held-out. One real reader request; the rest are drafts.\n- `shelv_vibe_model_comparison.ipynb`: self-contained Colab workflow.\n- `create_colab_notebook.py`: regenerate the notebook after code/query/README changes.\n- `test_*.py`: preparation, ranking, evaluation, and notebook portability checks.\n- `requirements.txt`: inference dependencies; experiment outputs record installed versions.\n\nRaw scrapers, one-off audits, historical results, download workarounds, and old instruction documents are not part of this export. They remain in the original Shelv workspace as evidence. Only this README is needed here.\n\n## Setup\n\nUse Python 3.10 or newer. From this project\'s directory:\n\n```sh\npython3 -m venv .venv\nsource .venv/bin/activate\npython -m pip install -r requirements.txt\npython -m unittest discover -s . -p \'test_*.py\' -v\n```\n\n## Dataset and current state\n\nPlace your catalog at `data/books_final.csv`, or pass `--input /path/to/file.csv`. The local export includes the 12,999-record final catalog with nine user-reviewed description updates. Git intentionally ignores data, generated results, model weights, and environments. A fresh clone therefore needs its own dataset copy. No old evaluation data is silently included.\n\nRequired CSV columns: `title,description,subjects,language,externalId,sourceName,sourceUrl`. Existing `record_id` values are preserved and checked for duplicates; otherwise IDs are derived from source fields and input row order. Preserve supplied IDs when filtering/reordering. Additional metadata columns pass through unchanged.\n\n**Data quality is not finished.** The current preparation rule only requires a nonblank title and description. It does not reject title-only text or placeholder strings. Quality flags in the final catalog remain metadata, not automatic eligibility rules. Before the next decision-making comparison, document a common eligibility policy and supply the same filtered CSV to all candidates. Do not treat the full catalog or the presence of a description as proof of mood evidence.\n\nEarlier work compared the four methods and collected human grades, but insufficient descriptions prevented a reliable overall winner. Existing historical results remain in the original project. After description/corpus changes, start a new experiment, rebuild vectors, and review changed evidence; do not transfer old rankings or grades unquestioningly.\n\n## Compare models in Colab\n\nUpload `shelv_vibe_model_comparison.ipynb` to Google Colab. Select a GPU runtime, run cells in order, and upload your chosen catalog CSV when requested. Download the results ZIP before closing Colab. Development grading and optional held-out evaluation are explained in the notebook. Each run captures code/data hashes, model revisions, installed packages, timings, and truncation counts.\n\nRegenerate the notebook after source changes:\n\n```sh\npython create_colab_notebook.py\npython -m unittest discover -s . -p \'test_*.py\' -v\n```\n\n## Compare models locally\n\n```sh\npython vibe_search.py prepare --input data/books_final.csv\npython benchmark.py run --candidate bm25 --output artifacts/comparison-v2\npython benchmark.py run --candidate e5-small --output artifacts/comparison-v2\npython benchmark.py run --candidate e5-base --output artifacts/comparison-v2\npython benchmark.py run --candidate bge-m3 --output artifacts/comparison-v2\npython benchmark.py pool --output artifacts/comparison-v2\n```\n\nCheck `python benchmark.py --help` for subcommands. Model downloads can be large; Colab was used for the previous comparison. Do not mix code, query, or corpus changes into an existing experiment folder.\n\nGrade `review_dev.csv` using descriptions as evidence: 2 = clear support, 1 = partial support, 0 = irrelevant/contradictory, U = insufficient evidence. Include evidence and reviewer for every numeric grade. Leave held-out judgments untouched until model and settings are frozen.\n\n```sh\npython benchmark.py evaluate --output artifacts/comparison-v2 --judgments artifacts/comparison-v2/review_dev.csv --split dev\n```\n\nPrecision@5 treats grade 2 as relevant. Graded nDCG@5 uses the judged candidate pool. The evaluator only scores queries whose complete pooled set has numeric grades; U and blank grades make a query incomplete. Report coverage alongside scores; a winner on one complete query is not an overall winner. Similarity is not a probability and there is no validated rejection threshold.\n\n## Single-model prototype\n\nE5-base is available for debugging, not a declared winner. Copy a full model commit SHA from the model repository and replace the placeholder:\n\n```sh\npython vibe_search.py prepare\npython vibe_search.py build --revision FULL_MODEL_COMMIT_SHA\npython vibe_search.py search "I need a book read peacefully" --k 5\n```\n\nModels embed title, subjects, and description; authors are excluded because source metadata mixes author/category values. Dense candidates use normalized vectors and exact dot-product ranking (cosine similarity), with a 512-token limit. E5 uses query/passage prefixes; BGE-M3 does not. The CLI reloads the model per invocation; a future Shelv service would keep it loaded.\n\n## Push to a new GitHub repository\n\nCreate an empty repository named `shelv-vibe-search` on GitHub (no generated README, license, or gitignore). In this directory:\n\n```sh\ngit init -b main\ngit add .\ngit status --short\ngit commit -m "Initialize Shelv vibe search project"\ngit remote add origin https://github.com/YOUR_USERNAME/shelv-vibe-search.git\ngit push -u origin main\n```\n\nReplace YOUR_USERNAME, or use the exact remote URL GitHub provides. You must authenticate to GitHub for the push. These commands follow [GitHub\'s import instructions](https://docs.github.com/en/migrations/importing-source-code/using-the-command-line-to-import-source-code/adding-locally-hosted-code-to-github). This export does not create a remote, choose repository visibility, or publish data.\n'}
for filename, content in EMBEDDED_FILES.items():
    (SCRIPTS / filename).write_text(content, encoding="utf-8")

def run_script(filename, *arguments):
    subprocess.run([sys.executable, str(SCRIPTS / filename), *map(str, arguments)], check=True)

def run_candidate(candidate):
    run_script("benchmark.py", "run", "--candidate", candidate,
               "--documents", PREPARED / "documents.json",
               "--queries", SCRIPTS / "benchmark_queries.json",
               "--output", EXPERIMENT, "--max-tokens", "512", "--batch-size", "8")

print("Experiment:", EXPERIMENT)


## 3. Upload the cleaned catalog and inspect coverage

Upload **books_final.csv**, not one individual source file. The local project copy
is in `data/book_preprocessing/output/books_final.csv`. Nothing in this notebook
modifies your local raw data. The search experiment retains only records with a
nonblank title and description; missing-description records are recorded separately.


In [ ]:
#@title Upload books_final.csv
from google.colab import files as colab_files

uploaded = colab_files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_names) != 1 or len(uploaded) != 1:
    raise ValueError("Please upload exactly one cleaned catalog CSV.")
INPUT = ROOT / "data/books_final.csv"
INPUT.parent.mkdir(parents=True, exist_ok=True)
INPUT.write_bytes(uploaded[csv_names[0]])
print("Uploaded", csv_names[0], "—", INPUT.stat().st_size, "bytes")


In [ ]:
#@title Prepare and measure the corpus
run_script("vibe_search.py", "prepare", "--input", INPUT, "--output", PREPARED)
profile = json.loads((PREPARED / "profile.json").read_text())
print("Input rows:", profile["input_rows"])
print("Eligible records:", profile["indexed_candidates"])
print("Excluded records:", profile["excluded_rows"])
assert profile["input_rows"] == profile["indexed_candidates"] + profile["excluded_rows"]


## 4. Inspect the queries and verify the software

There are 13 development queries and 8 held-out queries. Translation pairs stay in
the same split. Inspect the wording now; if you edit the query file, do so **before**
running candidates. Freeze the wording and judging rules before looking at results.

All dense candidates get the same text fields and a 512-token budget. Their tokenizers
can retain different amounts of text, so truncation is reported per model. This is a
controlled short-input pilot, not a test of BGE-M3's longer-context capability.


In [ ]:
#@title Show the frozen query set
queries = json.loads((SCRIPTS / "benchmark_queries.json").read_text())
for query in queries:
    print(query["id"], "|", query["split"], "|", query.get("origin", "draft_synthetic"), "|", query["query"])


In [ ]:
#@title Run correctness tests and save environment details
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(SCRIPTS), "-p", "test_*.py", "-v"], check=True)
packages = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
(EXPERIMENT / "environment-lock.txt").write_text(packages)
print("Tests verify mechanics, not whether book suggestions are relevant.")


## 5. Run the keyword baseline

BM25 matches words and weights them using their frequency in this catalog. It has no
translation or synonym model. It uses full descriptions with fixed, untuned parameters
(`k1=1.5`, `b=0.75`). It returns only positive-score matches. Treat it as a baseline,
not the strongest possible keyword-search implementation.


In [ ]:
#@title Run BM25
run_candidate("bm25")


## 6. Run multilingual E5-small

This is the smaller E5 candidate. The script uses the publisher's query/passage prefixes,
pins the exact model revision, and saves normalized vectors.
[Publisher documentation](https://huggingface.co/intfloat/multilingual-e5-small).


In [ ]:
#@title Run E5-small
run_candidate("e5-small")


## 7. Run multilingual E5-base

Use the identical catalog and queries. We test whether the larger E5 representation
improves relevance enough to justify its measured operating cost.
[Publisher documentation](https://huggingface.co/intfloat/multilingual-e5-base).


In [ ]:
#@title Run E5-base
run_candidate("e5-base")


## 8. Run BGE-M3 dense retrieval

This candidate does not use E5's query/passage prefixes. We compare only its dense-vector
retrieval, not its sparse/multi-vector modes or a reranker. Its token budget remains 512
for this experiment. [Publisher documentation](https://huggingface.co/BAAI/bge-m3).

Each model runs in a separate process, releasing its GPU memory when it exits.
If memory is insufficient, use a new experiment with a smaller batch size consistently
for every dense model. Do not mix old and new timing settings in one comparison.


In [ ]:
#@title Run BGE-M3
run_candidate("bge-m3")


## 9. Create blind review files and view measured performance

The pool is the union of each system's top 10 results for each query. Duplicate
query/record pairs are merged and shuffled. Reviewers see the query, title, original
description and source URL, but no model name, score or rank.

Performance can be measured immediately. Relevance cannot: blank judgments must stay
**Not judged**, never zero. Latency excludes model loading and uses three warm runs per
query; memory measurements describe this runtime, not a production guarantee.


In [ ]:
#@title Build review pools without overwriting any existing judgments
if not (EXPERIMENT / "review_dev.csv").exists() and not (EXPERIMENT / "review_test.csv").exists():
    run_script("benchmark.py", "pool", "--output", EXPERIMENT)
elif not all((EXPERIMENT / f"review_{split}.csv").exists() for split in ("dev", "test")):
    raise RuntimeError("Only one review file exists. Preserve it and inspect the experiment before continuing.")
else:
    print("Existing review files preserved.")
run_script("benchmark.py", "evaluate", "--output", EXPERIMENT,
           "--split", "dev", "--judgments", EXPERIMENT / "review_dev.csv")
from IPython.display import display, Markdown
display(Markdown((EXPERIMENT / "comparison_dev.md").read_text()))


## 10. Download your experiment

The ZIP contains the frozen corpus, query set, fingerprints, source code, model rankings,
timings, review CSVs and reports. It excludes downloaded model weights. Book vectors
are optional to keep the download smaller. Download before closing Colab: temporary
runtime storage can disappear. Nothing is automatically uploaded to Drive.

**Stop here for the first run.** Review `review_dev.csv` before using the optional cells below.


In [ ]:
#@title Download results
import zipfile
INCLUDE_BOOK_VECTORS = False  # Change to True if you also want the three .npy files.

def download_results():
    archive = Path("/content/shelv_vibe_comparison_results.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
        for path in EXPERIMENT.rglob("*"):
            if path.is_file() and (path.suffix != ".npy" or INCLUDE_BOOK_VECTORS):
                package.write(path, "experiment/" + str(path.relative_to(EXPERIMENT)))
        package.write(PREPARED / "profile.json", "data_profile.json")
        for name in EMBEDDED_FILES:
            package.write(SCRIPTS / name, "code/" + name)
    print("Download:", archive.name)
    colab_files.download(str(archive))

download_results()


## 11. Optional: calculate relevance after human review

Fill these columns in **review_dev.csv**, retaining query and record IDs:

| Grade | Meaning |
| --- | --- |
| 2 | Description clearly supports the requested mood/theme and constraints |
| 1 | Partial match; important aspects are unsupported |
| 0 | Irrelevant or contradicts the request |
| U | Insufficient evidence to judge |

Add a supporting excerpt or mismatch explanation in **evidence** and your name in
**reviewer**. Have a second reviewer grade a subset independently when possible.
The evaluator uses only fully judged query pools and never silently converts blank/U
grades into irrelevant judgments. Do not invent labels just to fill the table.

Precision@5 counts grade-2 results out of five. Pooled nDCG@5 rewards relevant results
appearing earlier, but its ideal ranking only covers the judged pool, not the full
catalog. Check language/intent slices and failures alongside the aggregate.


In [ ]:
#@title Upload reviewed development judgments (run only after review)
RUN_RELEVANCE_EVALUATION = False  # Set True when you have completed review_dev.csv.
if RUN_RELEVANCE_EVALUATION:
    reviewed = colab_files.upload()
    if len(reviewed) != 1:
        raise ValueError("Upload exactly one reviewed CSV.")
    reviewed_path = EXPERIMENT / "review_dev_judged.csv"
    reviewed_path.write_bytes(next(iter(reviewed.values())))
    run_script("benchmark.py", "evaluate", "--output", EXPERIMENT,
               "--split", "dev", "--judgments", reviewed_path)
    display(Markdown((EXPERIMENT / "comparison_dev.md").read_text()))
    download_results()
else:
    print("Skipped: human judgments have not been supplied.")


## 12. Optional: confirm on held-out queries

Choose a provisional candidate using **development** evidence and your latency/memory
constraints. Record that choice before reviewing `review_test.csv`. Grade the held-out
pool using the same rules, then run this cell. Do not tune on held-out results.

This small pilot (one user query plus draft cases) cannot establish broad user quality.
More real queries, reviewer agreement and adequate judgments are needed before a
production decision. There is intentionally no automatic “winner” based on similarity
scores or speed alone. The CSV's duplicate editions may also affect record-level metrics.


In [ ]:
#@title Record the development choice and evaluate held-out judgments
SELECTED_ON_DEV = ""  # bm25, e5-small, e5-base or bge-m3
SELECTION_REASON = ""  # Explain measured relevance and operating tradeoffs.
if SELECTED_ON_DEV:
    if SELECTED_ON_DEV not in {"bm25", "e5-small", "e5-base", "bge-m3"} or not SELECTION_REASON.strip():
        raise ValueError("Provide a known candidate and an evidence-based selection reason.")
    development = json.loads((EXPERIMENT / "evaluation_dev.json").read_text())
    if development["fully_judged_queries"] != development["total_queries"]:
        raise ValueError("Complete the development judgments before recording a selection.")
    selection_path = EXPERIMENT / "development_selection.json"
    selection = {"candidate": SELECTED_ON_DEV, "reason": SELECTION_REASON}
    if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
        raise ValueError("Selection already frozen. Do not change it after opening held-out results.")
    selection_path.write_text(json.dumps(selection, indent=2))
    reviewed = colab_files.upload()
    if len(reviewed) != 1:
        raise ValueError("Upload exactly one reviewed test CSV.")
    reviewed_path = EXPERIMENT / "review_test_judged.csv"
    reviewed_path.write_bytes(next(iter(reviewed.values())))
    run_script("benchmark.py", "evaluate", "--output", EXPERIMENT,
               "--split", "test", "--judgments", reviewed_path)
    display(Markdown((EXPERIMENT / "comparison_test.md").read_text()))
    download_results()
else:
    print("Skipped: no development-based candidate has been selected.")
